# 3.14 — Logistic Regression

Logistic regression is the simplest probabilistic classifier: it turns a linear score into a probability with the sigmoid link, trains that probability with empirical log loss, and chooses among model variants only after accounting for cost, regularization, validation gaps, and decision thresholds.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build logistic regression one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is visible, including the log-odds interpretation, empirical risk average, regularized decision score, gradients, and threshold behavior. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, exponentials, dot products, and hand-checkable losses.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for tiny data and training demos.

### 1. Linear score → log-odds → probability

Logistic regression starts exactly like a linear model: compute a score $z=w^\top x+b$. The difference is how we read that score. In logistic regression, $z$ is a **log-odds** value: positive means class 1 is more likely than class 0, negative means less likely, and zero means 50–50. The sigmoid function $\sigma(z)=1/(1+e^{-z})$ converts that unbounded score into a probability in $(0,1)$.

In [ ]:
z_grid_w = np.linspace(-8, 8, 201)  # sweep scores from strongly negative to strongly positive.
p_grid_w = 1 / (1 + np.exp(-z_grid_w))  # sigmoid converts scores into probabilities.
print("sigmoid(-2), sigmoid(0), sigmoid(2):", np.round(1 / (1 + np.exp(-np.array([-2., 0., 2.]))), 3))
assert np.allclose(np.round(1 / (1 + np.exp(-np.array([-2., 0., 2.]))), 3), [0.119, 0.5, 0.881])

▶ What you'll see: score 0 maps to 0.5, while equal positive and negative scores map symmetrically around 0.5.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.plot(z_grid_w, p_grid_w, color="teal")
plt.axvline(0, color="black", linewidth=0.8)
plt.axhline(0.5, color="gray", linestyle="--")
plt.title("1: sigmoid turns a score into probability")
plt.xlabel("linear score z = w·x + b")
plt.ylabel("p(y=1|x)")
plt.show()

▶ What you'll see: an S-shaped curve that is almost flat near 0 and 1 but steep near the decision boundary.

*Why it's done this way:* a raw linear score can be any real number, but a class probability must stay between 0 and 1. The sigmoid is chosen because it is exactly the inverse of the log-odds transform: if $p=\sigma(z)$, then $\log(p/(1-p))=z$. That makes each weight an additive change in log-odds while keeping the output probabilistic.

### 2. The decision boundary is linear even though probabilities are curved

The probability curve is nonlinear, but the boundary for a 0.5 threshold is still linear. Since $\sigma(0)=0.5$, predicting class 1 at threshold 0.5 is equivalent to checking whether $w^\top x+b\ge 0$. In two dimensions, that equation is a line.

In [ ]:
X_w = np.array([[0.2, 1.2], [0.8, 1.0], [1.2, 0.8], [1.8, 0.6], [2.2, 0.4], [2.6, 0.2]])
y_w = np.array([0, 0, 0, 1, 1, 1])
w_w = np.array([2.0, -1.0])
b_w = -2.0
z_w = X_w @ w_w + b_w
p_w = 1 / (1 + np.exp(-z_w))
print("scores:", np.round(z_w, 2))
print("probabilities:", np.round(p_w, 3))

▶ What you'll see: points with positive scores have probabilities above 0.5, and negative scores fall below 0.5.

In [ ]:
x1_line_w = np.linspace(0, 3, 100)
x2_line_w = (w_w[0] * x1_line_w + b_w) / (-w_w[1])  # solve w1*x1 + w2*x2 + b = 0 for x2.
plt.figure(figsize=(4.6, 3.6))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=80)
plt.plot(x1_line_w, x2_line_w, color="black", label="p=0.5 boundary")
plt.xlim(0, 3); plt.ylim(0, 1.6)
plt.xlabel("x1"); plt.ylabel("x2"); plt.title("2: linear boundary, logistic probabilities")
plt.legend(); plt.show()

▶ What you'll see: a straight line separates the two classes; the sigmoid only changes how confident each side becomes.

*Why it's done this way:* logistic regression keeps the interpretability and geometry of a linear classifier, then wraps that line with calibrated probability values. This is why it can make both hard decisions and softer risk estimates.

### 3. Log loss is the empirical risk logistic regression optimizes

A probability model should be rewarded for assigning high probability to the true label and punished for confident mistakes. For a binary label $y\in\{0,1\}$ and predicted probability $p$, the per-example log loss is $-[y\log p+(1-y)\log(1-p)]$. The empirical risk $R_S$ is the average of those losses over the training sample.

In [ ]:
losses_w = np.array([0.235, 0.109, 0.471])  # verified toy losses from the lesson prose.
R_S_w = float(np.mean(losses_w))
print("toy losses:", losses_w)
print("empirical risk R_S:", round(R_S_w, 3))
assert round(R_S_w, 3) == 0.272

▶ What you'll see: the three losses average to 0.272, the raw training quantity in the lesson.

In [ ]:
p_line_w = np.linspace(0.01, 0.99, 200)
loss_y1_w = -np.log(p_line_w)
loss_y0_w = -np.log(1 - p_line_w)
plt.figure(figsize=(4.6, 3.2))
plt.plot(p_line_w, loss_y1_w, label="true y=1", color="teal")
plt.plot(p_line_w, loss_y0_w, label="true y=0", color="orange")
plt.title("3: log loss punishes confident mistakes")
plt.xlabel("predicted p(y=1)"); plt.ylabel("loss")
plt.legend(); plt.show()

▶ What you'll see: if the true label is 1, loss explodes near p=0; if the true label is 0, it explodes near p=1.

*Why it's done this way:* log loss is the negative log-likelihood of Bernoulli labels. Averaging it is empirical risk minimization: each example contributes one probability-quality score, and training tries to lower their mean. The curve's steep penalty near confident errors discourages brittle, overconfident classifiers.

### 4. Add cost or regularization before selecting a model

The lesson warns that the raw training score is not the whole decision. If the method carries a complexity, regularization, or operational cost of 0.070, the decision score is $R_S+cost$, not $R_S$ alone. That extra term is how we keep a flattering fit from being mistaken for a durable model.

In [ ]:
cost_w = 0.070
score_w = round(R_S_w + cost_w, 3)  # use the rounded lesson score for later verified gap arithmetic.
print("raw R_S:", round(R_S_w, 3), "cost:", round(cost_w, 3), "decision score:", round(score_w, 3))
assert round(score_w, 3) == 0.342

▶ What you'll see: the selection score is 0.342, not the prettier raw average 0.272.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["R_S", "cost", "R_S + cost"], [R_S_w, cost_w, score_w], color=["steelblue", "orange", "seagreen"])
plt.title("4: selection uses the full score")
plt.ylabel("score component")
plt.show()

▶ What you'll see: the added cost is small but large enough to change the number used for model selection.

*Why it's done this way:* empirical risk estimates current-sample fit, while a cost term encodes the price of flexibility or deployment. Optimizing only the raw term rewards models that can exploit sample quirks; adding cost makes the comparison closer to the future decision we actually care about.

### 5. Compare alternatives by gaps, not isolated scores

A more flexible alternative has decision score 0.394. The baseline score is 0.342, so the absolute gap is 0.052 and the relative gap is about 13.2%. The lower score wins here, but the size of the gap tells us how much confidence that preference deserves.

In [ ]:
alt_score_w = 0.394
gap_w = alt_score_w - score_w
rel_gap_w = gap_w / alt_score_w
print("baseline:", round(score_w, 3), "alternative:", alt_score_w)
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3))
assert round(gap_w, 3) == 0.052
assert round(rel_gap_w, 3) == 0.132

▶ What you'll see: the baseline beats the alternative, but the margin is finite and should be judged against uncertainty.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["baseline", "flexible alt"], [score_w, alt_score_w], color=["seagreen", "crimson"])
plt.title("5: model comparison on one scale")
plt.ylabel("decision score (lower is better)")
plt.show()

▶ What you'll see: both bars are on the same score scale, making the comparison meaningful.

*Why it's done this way:* a probability, a loss, and a penalized score are not interchangeable. Logistic regression should be compared using the same selection scale for every candidate; the gap then measures the evidence for preferring one setting over another.

### 6. Regularized gradient descent learns the weights

Training logistic regression is not magic. For predictions $p=\sigma(Xw+b)$, the average log-loss gradient is $X^\top(p-y)/n$ for weights and $mean(p-y)$ for the bias. L2 regularization adds $\lambda w$ to the weight gradient, shrinking large coefficients unless they earn their keep by lowering loss.

In [ ]:
Xg_w = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 0.5], [3.0, 0.2]])
yg_w = np.array([0., 0., 1., 1.])
wg_w = np.array([0.2, -0.1])
bg_w = -0.2
pg_w = 1 / (1 + np.exp(-(Xg_w @ wg_w + bg_w)))
print("initial probabilities:", np.round(pg_w, 3))

▶ What you'll see: the untrained probabilities are only weakly separated between the two classes.

In [ ]:
lam_w = 0.1
grad_w_w = Xg_w.T @ (pg_w - yg_w) / len(yg_w) + lam_w * wg_w
grad_b_w = float(np.mean(pg_w - yg_w))
print("weight gradient:", np.round(grad_w_w, 3), "bias gradient:", round(grad_b_w, 3))
assert np.allclose(np.round(grad_w_w, 3), [-0.397, 0.137])

▶ What you'll see: the first weight gradient is negative, so gradient descent will increase that weight.

In [ ]:
eta_w = 0.5
wg_new_w = wg_w - eta_w * grad_w_w
bg_new_w = bg_w - eta_w * grad_b_w
pg_new_w = 1 / (1 + np.exp(-(Xg_w @ wg_new_w + bg_new_w)))
print("new weights:", np.round(wg_new_w, 3), "new bias:", round(bg_new_w, 3))
print("new probabilities:", np.round(pg_new_w, 3))
assert np.allclose(np.round(wg_new_w, 3), [0.399, -0.169])

▶ What you'll see: one gradient step increases class-1 probabilities for the larger-x examples.

*Why it's done this way:* the derivative $p-y$ is the signed probability error. Multiplying by features tells each weight whether increasing that feature's coefficient would reduce average log loss. The regularization term pulls weights back toward zero, so each coefficient must justify its size with predictive value.

### 7. Thresholds turn probabilities into decisions

The model outputs probabilities, but applications need actions. A threshold converts $p$ into a class label: predict 1 if $p\ge t$. Moving $t$ changes false positives and false negatives without retraining the model, which is why evaluation lessons revisit threshold choice.

In [ ]:
probs_w = np.array([0.08, 0.22, 0.41, 0.55, 0.73, 0.91])
truth_w = np.array([0, 0, 1, 0, 1, 1])
for t_w in [0.3, 0.5, 0.7]:
    pred_t_w = (probs_w >= t_w).astype(int)
    acc_t_w = np.mean(pred_t_w == truth_w)
    print("threshold", t_w, "predictions", pred_t_w, "accuracy", round(acc_t_w, 3))

▶ What you'll see: the same probabilities produce different predicted labels as the threshold moves.

In [ ]:
thresholds_w = np.linspace(0.1, 0.9, 17)
accs_w = []
for t_w in thresholds_w:
    accs_w.append(np.mean((probs_w >= t_w).astype(int) == truth_w))
plt.figure(figsize=(4.6, 3))
plt.plot(thresholds_w, accs_w, marker="o", color="purple")
plt.title("7: threshold changes decisions")
plt.xlabel("threshold"); plt.ylabel("accuracy on tiny set")
plt.ylim(0, 1.05); plt.show()

▶ What you'll see: accuracy is piecewise flat because predictions change only when the threshold crosses a probability value.

*Why it's done this way:* probabilities separate modeling from decision-making. The sigmoid/log-loss training estimates risk; a threshold then encodes the downstream cost tradeoff, such as whether false positives or false negatives are more expensive.

### 8. Stabilization can win the end-to-end score

The lesson's stabilizing knob reduces the baseline decision score by 20%, giving $0.80\cdot0.342=0.274$. The final choice compares the baseline, the flexible alternative, and the stabilized score on the same scale; the smallest score is carried forward.

In [ ]:
stable_score_w = 0.80 * score_w
scores_all_w = np.array([score_w, alt_score_w, stable_score_w])
labels_all_w = np.array(["baseline", "flexible", "stabilized"])
best_idx_w = int(np.argmin(scores_all_w))
print("stabilized score:", round(stable_score_w, 3))
print("winner:", labels_all_w[best_idx_w], "score", round(scores_all_w[best_idx_w], 3))
assert round(stable_score_w, 3) == 0.274
assert labels_all_w[best_idx_w] == "stabilized"

▶ What you'll see: the stabilized version reaches the lowest verified toy score, 0.274.

In [ ]:
plt.figure(figsize=(4.8, 3))
colors_w = ["gray", "crimson", "seagreen"]
plt.bar(labels_all_w, scores_all_w, color=colors_w)
plt.title("8: final model selection score")
plt.ylabel("lower is better")
plt.show()

▶ What you'll see: the stabilized bar is lowest, even though stabilization may have constrained the model.

*Why it's done this way:* regularization and stability controls intentionally give up some flexibility. If that constraint reduces future-facing decision score, it is not a compromise in quality — it is the mechanism by which the model becomes less brittle.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, vectorized sigmoid calculations, losses, gradients, and assertions.
import matplotlib.pyplot as plt # load Matplotlib for compact curves, bars, scatters, and heatmaps.
np.random.seed(0) # make every randomized example reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute a linear log-odds score

**Goal.** Combine features with weights into one score, because logistic regression begins with the same linear ingredient as a linear classifier. We build it in 2 steps.

In [ ]:
x_b1 = np.array([2.0, 0.5]) # define one example with two features.
w_b1 = np.array([1.2, -0.4]) # define feature weights that add evidence and subtract evidence.
b_b1 = -1.0 # define the intercept as baseline log-odds before seeing features.
print("x:", x_b1, "w:", w_b1, "b:", b_b1) # inspect ingredients before taking the dot product.

▶ What you'll see: two features, two weights, and one intercept ready to be combined.

In [ ]:
z_b1 = float(w_b1 @ x_b1 + b_b1) # compute z = w^T x + b as the log-odds score.
print("log-odds score z:", round(z_b1, 3)) # inspect the scalar score.
assert round(z_b1, 3) == 1.2 # verify 1.2*2 + (-0.4)*0.5 - 1 = 1.2.
plt.figure(figsize=(4, 3)) # create a compact contribution plot.
plt.bar(["w1*x1", "w2*x2", "b"], [w_b1[0]*x_b1[0], w_b1[1]*x_b1[1], b_b1], color=["teal", "orange", "gray"]) # show the terms that sum to z.
plt.title("Basic 1: pieces of the log-odds score") # title the diagnostic plot.
plt.ylabel("contribution to z") # label the contribution scale.
plt.show() # display the chart.

▶ What you'll see: the positive first feature dominates the negative feature and intercept, leaving z = 1.2.

👀 Takeaway: logistic regression is linear in log-odds before the sigmoid turns the score into a probability.

### Basic 2 — Apply the sigmoid by hand

**Goal.** Convert a score into a probability, because classification decisions should be based on values between 0 and 1. We build it in 2 steps.

In [ ]:
z_b2 = 1.2 # reuse the worked log-odds score from the previous example.
exp_neg_b2 = np.exp(-z_b2) # compute e^{-z}, the denominator's nonlinear part.
print("exp(-z):", round(exp_neg_b2, 3)) # inspect the exponential before forming the probability.

▶ What you'll see: a positive score makes exp(-z) smaller than 1.

In [ ]:
p_b2 = 1 / (1 + exp_neg_b2) # compute sigmoid(z) = 1/(1+exp(-z)).
print("probability p(y=1|x):", round(p_b2, 3)) # inspect the probability.
assert round(p_b2, 3) == 0.769 # verify sigmoid(1.2).
plt.figure(figsize=(4, 3)) # create a one-point probability chart.
plt.bar(["p(class 0)", "p(class 1)"], [1 - p_b2, p_b2], color=["orange", "teal"]) # show complementary class probabilities.
plt.ylim(0, 1) # keep probability scale fixed.
plt.title("Basic 2: sigmoid probability") # title the plot.
plt.show() # display the plot.

▶ What you'll see: class 1 receives probability about 0.769, so class 0 receives about 0.231.

👀 Takeaway: sigmoid converts unbounded evidence into a normalized binary probability.

### Basic 3 — Interpret odds and log-odds

**Goal.** Move between probability, odds, and log-odds, because logistic regression weights add on the log-odds scale. We build it in 3 steps.

In [ ]:
p_b3 = 0.8 # choose an easy probability for odds interpretation.
odds_b3 = p_b3 / (1 - p_b3) # convert probability to odds of class 1 versus class 0.
print("probability:", p_b3, "odds:", round(odds_b3, 3)) # inspect p and odds side by side.
assert round(odds_b3, 3) == 4.0 # verify 0.8 / 0.2 = 4.

▶ What you'll see: probability 0.8 means the odds favor class 1 by 4 to 1.

In [ ]:
log_odds_b3 = np.log(odds_b3) # take the natural log so odds become an additive score.
back_to_p_b3 = 1 / (1 + np.exp(-log_odds_b3)) # invert log-odds with the sigmoid.
print("log-odds:", round(log_odds_b3, 3), "back to p:", round(back_to_p_b3, 3)) # verify the two transforms agree.
assert round(back_to_p_b3, 3) == 0.8 # verify inverse relationship.

▶ What you'll see: log-odds 1.386 maps right back to probability 0.8.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact transformation plot.
plt.bar(["p", "odds", "log-odds"], [p_b3, odds_b3, log_odds_b3], color=["teal", "orange", "purple"]) # compare the scales.
plt.title("Basic 3: probability vs odds scales") # title the scale comparison.
plt.show() # display the chart.

▶ What you'll see: the same belief looks different on probability, odds, and log-odds scales.

👀 Takeaway: logistic regression is linear in log-odds, not directly linear in probability.

### Basic 4 — Predict labels with a threshold

**Goal.** Turn probabilities into class labels, because applications need decisions after the model estimates risk. We build it in 2 steps.

In [ ]:
probs_b4 = np.array([0.12, 0.49, 0.51, 0.83]) # define four predicted class-1 probabilities.
threshold_b4 = 0.5 # use the default symmetric decision threshold.
print("probabilities:", probs_b4) # inspect probabilities before thresholding.

▶ What you'll see: two probabilities sit below 0.5 and two sit at or above 0.5.

In [ ]:
preds_b4 = (probs_b4 >= threshold_b4).astype(int) # classify as 1 when probability crosses the threshold.
print("predicted labels:", preds_b4) # inspect hard predictions.
assert np.array_equal(preds_b4, np.array([0, 0, 1, 1])) # verify threshold logic.
plt.figure(figsize=(4, 3)) # create a threshold visualization.
plt.bar(np.arange(len(probs_b4)), probs_b4, color="steelblue") # plot probabilities.
plt.axhline(threshold_b4, color="red", linestyle="--", label="threshold") # draw the decision cutoff.
plt.ylim(0, 1); plt.title("Basic 4: thresholded probabilities"); plt.legend(); plt.show() # finish and display the chart.

▶ What you'll see: bars above the red line are predicted as class 1.

👀 Takeaway: thresholding is a decision rule layered on top of probability estimation.

### Basic 5 — Compute one log-loss value

**Goal.** Score one predicted probability against its true label, because logistic regression learns by minimizing log loss. We build it in 2 steps.

In [ ]:
y_b5 = 1 # set the true label to positive.
p_b5 = 0.8 # set the model's predicted probability for the positive class.
print("label:", y_b5, "predicted probability:", p_b5) # inspect the one-example prediction.

▶ What you'll see: the model is fairly confident in the correct class.

In [ ]:
loss_b5 = -(y_b5 * np.log(p_b5) + (1 - y_b5) * np.log(1 - p_b5)) # compute binary cross-entropy.
print("log loss:", round(loss_b5, 3)) # inspect the penalty.
assert round(loss_b5, 3) == 0.223 # verify -log(0.8).
plt.figure(figsize=(4, 3)) # create a one-example loss bar.
plt.bar(["loss"], [loss_b5], color="crimson") # show the penalty magnitude.
plt.title("Basic 5: one-example log loss") # title the plot.
plt.ylim(0, 1); plt.show() # display the chart.

▶ What you'll see: a correct 0.8 prediction receives a modest loss of about 0.223.

👀 Takeaway: log loss is small for high probability on the true class and large for confident mistakes.

### Basic 6 — Average losses into empirical risk

**Goal.** Average per-example losses, because empirical risk minimization optimizes a sample mean. We build it in 2 steps.

In [ ]:
losses_b6 = np.array([0.235, 0.109, 0.471]) # use the verified lesson losses.
print("losses:", losses_b6) # inspect the sample losses before averaging.

▶ What you'll see: three per-example penalties that will form one training score.

In [ ]:
risk_b6 = float(np.mean(losses_b6)) # average losses to estimate empirical risk.
print("empirical risk:", round(risk_b6, 3)) # inspect the training objective value.
assert round(risk_b6, 3) == 0.272 # verify the lesson arithmetic.
plt.figure(figsize=(4, 3)) # create a compact loss plot.
plt.bar(["ex1", "ex2", "ex3"], losses_b6, color="teal") # show each loss.
plt.axhline(risk_b6, color="black", linestyle="--", label="mean") # show the average.
plt.title("Basic 6: empirical risk is an average"); plt.legend(); plt.show() # display the chart.

▶ What you'll see: the dashed mean line summarizes the three individual losses.

👀 Takeaway: the raw training score is an average over examples, not a single hand-picked case.

### Basic 7 — Add a selection cost

**Goal.** Add cost to empirical risk, because model selection should include complexity or operational penalties. We build it in 2 steps.

In [ ]:
risk_b7 = 0.272 # use the rounded empirical risk from the lesson.
cost_b7 = 0.070 # use the lesson's method cost.
print("risk:", risk_b7, "cost:", cost_b7) # inspect the two score components.

▶ What you'll see: the prettier raw number is only one part of the score.

In [ ]:
score_b7 = risk_b7 + cost_b7 # combine fit and cost for selection.
print("decision score:", round(score_b7, 3)) # inspect the full score.
assert round(score_b7, 3) == 0.342 # verify the lesson score.
plt.figure(figsize=(4, 3)) # create a score breakdown plot.
plt.bar(["risk", "cost", "total"], [risk_b7, cost_b7, score_b7], color=["steelblue", "orange", "seagreen"]) # show additive components.
plt.title("Basic 7: fit plus cost") # title the score plot.
plt.show() # display the chart.

▶ What you'll see: total score rises from 0.272 to 0.342 after including cost.

👀 Takeaway: selecting by raw training risk alone silently ignores the cost term.

### Basic 8 — Compare two candidate scores

**Goal.** Compute absolute and relative gaps, because a model comparison is only meaningful on a shared scale. We build it in 2 steps.

In [ ]:
base_b8 = 0.342 # baseline decision score from the lesson.
alt_b8 = 0.394 # flexible alternative decision score from the lesson.
print("baseline:", base_b8, "alternative:", alt_b8) # inspect the candidates.

▶ What you'll see: the baseline has the smaller decision score.

In [ ]:
gap_b8 = alt_b8 - base_b8 # compute the absolute evidence gap.
rel_b8 = gap_b8 / alt_b8 # compute the gap relative to the alternative score.
print("gap:", round(gap_b8, 3), "relative gap:", round(rel_b8, 3)) # inspect comparison strength.
assert round(gap_b8, 3) == 0.052 # verify lesson gap.
assert round(rel_b8, 3) == 0.132 # verify relative gap.
plt.figure(figsize=(4, 3)) # create a candidate comparison plot.
plt.bar(["baseline", "alternative"], [base_b8, alt_b8], color=["seagreen", "crimson"]) # compare scores.
plt.title("Basic 8: lower decision score wins") # title the plot.
plt.ylabel("score") # label score scale.
plt.show() # display the chart.

▶ What you'll see: the difference is visible but modest, so uncertainty still matters.

👀 Takeaway: gaps quantify how strong a model preference is, not just which number is lower.

### Basic 9 — Apply a stabilizing reduction

**Goal.** Apply the lesson's 20% stabilization improvement, because constrained models can sometimes make better future decisions. We build it in 2 steps.

In [ ]:
score_b9 = 0.342 # start from the baseline decision score.
multiplier_b9 = 0.80 # encode a 20% reduction as retaining 80% of the score.
print("score:", score_b9, "multiplier:", multiplier_b9) # inspect the stabilization setup.

▶ What you'll see: stabilization is represented as a simple multiplicative score change.

In [ ]:
stable_b9 = multiplier_b9 * score_b9 # compute the stabilized decision score.
print("stabilized score:", round(stable_b9, 3)) # inspect the new score.
assert round(stable_b9, 3) == 0.274 # verify 0.80*0.342.
plt.figure(figsize=(4, 3)) # create a before-after chart.
plt.bar(["before", "stabilized"], [score_b9, stable_b9], color=["gray", "teal"]) # show the reduction.
plt.title("Basic 9: stabilization lowers score") # title the plot.
plt.ylabel("score") # label score axis.
plt.show() # display the chart.

▶ What you'll see: the stabilized score is visibly lower than the original baseline score.

👀 Takeaway: regularization is useful when the final decision score improves, not merely because it sounds safer.

### Basic 10 — Choose the minimum final score

**Goal.** Select the winning model from baseline, flexible, and stabilized candidates, because model selection should use the full decision score. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.342, 0.394, 0.274]) # collect baseline, flexible alternative, and stabilized scores.
names_b10 = np.array(["baseline", "flexible", "stabilized"]) # label the candidates.
print("scores:", dict(zip(names_b10, scores_b10))) # inspect all candidates on the same scale.

▶ What you'll see: all three candidates are comparable because they are decision scores.

In [ ]:
winner_b10 = names_b10[int(np.argmin(scores_b10))] # choose the lowest score.
print("winner:", winner_b10, "score:", round(float(np.min(scores_b10)), 3)) # inspect the selected model.
assert winner_b10 == "stabilized" # verify the lesson's final toy choice.
plt.figure(figsize=(4.5, 3)) # create a final selection plot.
plt.bar(names_b10, scores_b10, color=["gray", "crimson", "seagreen"]) # show all scores.
plt.title("Basic 10: final decision") # title the selection plot.
plt.ylabel("lower is better") # label the objective direction.
plt.show() # display the chart.

▶ What you'll see: the stabilized candidate is the lowest bar at 0.274.

👀 Takeaway: the correct winner is the model with the best full score, not the prettiest training fragment.

## 🟡 Easy

### Easy 1 — Vectorize logistic predictions for a tiny dataset

**Goal.** Compute scores and probabilities for several examples at once, because real logistic regression trains on matrices rather than one row at a time. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 0.5], [3.0, 0.2]]) # create four examples with two features.
y_e1 = np.array([0, 0, 1, 1]) # define binary labels for the examples.
w_e1 = np.array([1.0, -0.5]) # choose a simple weight vector.
b_e1 = -1.0 # choose an intercept.
print("X shape:", X_e1.shape, "labels:", y_e1) # inspect data dimensions and targets.

▶ What you'll see: a 4×2 feature matrix and four binary labels.

In [ ]:
z_e1 = X_e1 @ w_e1 + b_e1 # compute all linear scores in one matrix-vector product.
p_e1 = 1 / (1 + np.exp(-z_e1)) # convert scores to probabilities.
print("scores:", np.round(z_e1, 3)) # inspect linear scores.
print("probabilities:", np.round(p_e1, 3)) # inspect sigmoid outputs.
assert np.allclose(np.round(p_e1, 3), [0.182, 0.378, 0.679, 0.870]) # verify vectorized probabilities.

▶ What you'll see: probability rises as the first feature becomes larger.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a compact probability plot.
plt.scatter(X_e1[:, 0], p_e1, c=y_e1, cmap="coolwarm", s=90) # plot probability against the most informative feature.
plt.axhline(0.5, color="gray", linestyle="--") # show the default decision threshold.
plt.title("Easy 1: vectorized logistic predictions") # title the figure.
plt.xlabel("feature x1"); plt.ylabel("p(y=1|x)"); plt.show() # label and display.

▶ What you'll see: examples with larger x1 cross above the 0.5 threshold.

👀 Takeaway: logistic regression applies the same sigmoid rule to every row of a design matrix.

### Easy 2 — Compute dataset log loss from probabilities

**Goal.** Average binary cross-entropy over a small dataset, because this is the empirical risk that training tries to reduce. We build it in 3 steps.

In [ ]:
p_e2 = np.array([0.182, 0.378, 0.679, 0.870]) # reuse rounded probabilities from a tiny classifier.
y_e2 = np.array([0, 0, 1, 1]) # define true labels.
eps_e2 = 1e-12 # define a tiny clipping value to avoid log(0) in general.
print("p:", p_e2, "y:", y_e2) # inspect predictions and labels.

▶ What you'll see: negative examples have lower probabilities than positive examples.

In [ ]:
p_clip_e2 = np.clip(p_e2, eps_e2, 1 - eps_e2) # clip probabilities for numerical safety.
losses_e2 = -(y_e2 * np.log(p_clip_e2) + (1 - y_e2) * np.log(1 - p_clip_e2)) # compute per-example log losses.
risk_e2 = float(np.mean(losses_e2)) # average the losses.
print("losses:", np.round(losses_e2, 3)) # inspect per-example penalties.
print("mean log loss:", round(risk_e2, 3)) # inspect empirical risk.
assert round(risk_e2, 3) == 0.301 # verify the worked mean.

▶ What you'll see: all four examples have moderate losses, averaging to about 0.301.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a per-example loss plot.
plt.bar(np.arange(len(losses_e2)), losses_e2, color="purple") # plot each loss.
plt.axhline(risk_e2, color="black", linestyle="--", label="mean") # mark the mean risk.
plt.title("Easy 2: per-example log losses") # title the figure.
plt.xlabel("example"); plt.ylabel("log loss"); plt.legend(); plt.show() # label and display.

▶ What you'll see: the dashed empirical risk summarizes the individual penalties.

👀 Takeaway: training loss is a mean over examples, so one confident error can raise the whole score.

### Easy 3 — Take one regularized gradient step

**Goal.** Update weights and bias once using the logistic gradient, because repeated steps are how the model learns. We build it in 4 steps.

In [ ]:
X_e3 = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 0.5], [3.0, 0.2]]) # define a tiny training matrix.
y_e3 = np.array([0., 0., 1., 1.]) # define labels as floats for gradient arithmetic.
w_e3 = np.array([0.2, -0.1]) # initialize weights.
b_e3 = -0.2 # initialize bias.
print("initial w:", w_e3, "b:", b_e3) # inspect starting parameters.

▶ What you'll see: small initial weights before learning.

In [ ]:
p_e3 = 1 / (1 + np.exp(-(X_e3 @ w_e3 + b_e3))) # compute current probabilities.
loss_e3 = float(np.mean(-(y_e3*np.log(p_e3) + (1-y_e3)*np.log(1-p_e3)))) # compute current unregularized log loss.
print("initial probabilities:", np.round(p_e3, 3)) # inspect model outputs.
print("initial loss:", round(loss_e3, 3)) # inspect starting loss.

▶ What you'll see: outputs are not yet confident, and loss is above the later value.

In [ ]:
lam_e3 = 0.1 # set L2 regularization strength.
eta_e3 = 0.5 # set learning rate.
grad_w_e3 = X_e3.T @ (p_e3 - y_e3) / len(y_e3) + lam_e3 * w_e3 # compute regularized weight gradient.
grad_b_e3 = float(np.mean(p_e3 - y_e3)) # compute bias gradient without regularizing the intercept.
print("grad_w:", np.round(grad_w_e3, 3), "grad_b:", round(grad_b_e3, 3)) # inspect descent direction.
assert np.allclose(np.round(grad_w_e3, 3), [-0.397, 0.137]) # verify the worked gradient.

▶ What you'll see: the first feature weight should increase because its gradient is negative.

In [ ]:
w_new_e3 = w_e3 - eta_e3 * grad_w_e3 # descend on weights.
b_new_e3 = b_e3 - eta_e3 * grad_b_e3 # descend on bias.
p_new_e3 = 1 / (1 + np.exp(-(X_e3 @ w_new_e3 + b_new_e3))) # compute updated probabilities.
loss_new_e3 = float(np.mean(-(y_e3*np.log(p_new_e3) + (1-y_e3)*np.log(1-p_new_e3)))) # compute updated loss.
print("new w:", np.round(w_new_e3, 3), "new b:", round(b_new_e3, 3)) # inspect updated parameters.
print("new loss:", round(loss_new_e3, 3), "old loss:", round(loss_e3, 3)) # compare losses.
assert loss_new_e3 < loss_e3 # verify one step improved the unregularized fit.
plt.figure(figsize=(4, 3)); plt.bar(["before", "after"], [loss_e3, loss_new_e3], color=["gray", "teal"]); plt.title("Easy 3: one gradient step lowers loss"); plt.ylabel("log loss"); plt.show() # visualize improvement.

▶ What you'll see: the loss drops after one regularized gradient step.

👀 Takeaway: logistic gradients convert probability errors into parameter updates that reduce empirical risk.

### Easy 4 — Fit a tiny logistic model with gradient descent

**Goal.** Repeat gradient steps and watch loss decrease, because convergence is the practical training loop behind logistic regression. We build it in 4 steps.

In [ ]:
X_e4 = np.array([[0.0, 1.0], [0.5, 1.2], [1.0, 0.8], [2.0, 0.4], [2.5, 0.2], [3.0, 0.1]]) # create a separable-ish toy dataset.
y_e4 = np.array([0., 0., 0., 1., 1., 1.]) # define binary labels.
w_e4 = np.zeros(2) # start weights at zero.
b_e4 = 0.0 # start bias at zero.
print("examples:", len(y_e4)) # inspect sample size.

▶ What you'll see: six labeled points for a tiny training run.

In [ ]:
losses_e4 = [] # store loss values through training.
for step_e4 in range(400): # run many small gradient steps.
    z_e4 = X_e4 @ w_e4 + b_e4 # compute current scores.
    p_e4 = 1 / (1 + np.exp(-z_e4)) # compute probabilities.
    loss_e4 = float(np.mean(-(y_e4*np.log(p_e4) + (1-y_e4)*np.log(1-p_e4))) + 0.5 * 0.05 * np.sum(w_e4**2)) # compute regularized objective.
    losses_e4.append(loss_e4) # record objective.
    grad_w_e4 = X_e4.T @ (p_e4 - y_e4) / len(y_e4) + 0.05 * w_e4 # compute weight gradient.
    grad_b_e4 = float(np.mean(p_e4 - y_e4)) # compute bias gradient.
    w_e4 -= 0.2 * grad_w_e4 # update weights.
    b_e4 -= 0.2 * grad_b_e4 # update bias.
print("loss start -> end:", round(losses_e4[0], 3), "->", round(losses_e4[-1], 3)) # inspect convergence.
assert losses_e4[-1] < losses_e4[0] # verify training reduced the objective.

▶ What you'll see: the regularized objective falls substantially from its initial value.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a learning curve figure.
plt.plot(losses_e4, color="teal") # draw objective over steps.
plt.title("Easy 4: logistic training curve") # title the plot.
plt.xlabel("step"); plt.ylabel("regularized log loss"); plt.show() # label and display.

▶ What you'll see: a fast early drop followed by a plateau as the model converges.

In [ ]:
p_final_e4 = 1 / (1 + np.exp(-(X_e4 @ w_e4 + b_e4))) # compute final probabilities.
print("final probabilities:", np.round(p_final_e4, 3)) # inspect learned separation.
plt.figure(figsize=(4.6, 3)); plt.scatter(X_e4[:, 0], p_final_e4, c=y_e4, cmap="coolwarm", s=90); plt.axhline(0.5, color="gray", linestyle="--"); plt.title("Easy 4: learned probabilities"); plt.xlabel("x1"); plt.ylabel("p(y=1)"); plt.show() # visualize final probabilities.

▶ What you'll see: positive examples receive probabilities above the threshold and negatives below it.

👀 Takeaway: repeated gradient descent turns the logistic objective into a trained probabilistic classifier.

### Easy 5 — Evaluate threshold accuracy on held-out data

**Goal.** Compare thresholds on validation data, because the best probability model still needs an application-specific decision cutoff. We build it in 3 steps.

In [ ]:
val_p_e5 = np.array([0.10, 0.35, 0.48, 0.62, 0.76, 0.88]) # define held-out predicted probabilities.
val_y_e5 = np.array([0, 0, 1, 0, 1, 1]) # define held-out labels.
thresholds_e5 = np.array([0.3, 0.5, 0.7]) # compare three possible cutoffs.
print("validation probabilities:", val_p_e5) # inspect validation predictions.

▶ What you'll see: several probabilities near the middle, where threshold choice matters.

In [ ]:
accs_e5 = [] # store accuracy for each threshold.
for t_e5 in thresholds_e5: # test each cutoff.
    pred_e5 = (val_p_e5 >= t_e5).astype(int) # make thresholded predictions.
    accs_e5.append(np.mean(pred_e5 == val_y_e5)) # compute validation accuracy.
print("accuracies:", np.round(accs_e5, 3)) # inspect threshold performance.
assert np.allclose(np.round(accs_e5, 3), [0.667, 0.667, 0.833]) # verify worked accuracies.

▶ What you'll see: threshold 0.7 performs best on this tiny validation set.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a threshold comparison plot.
plt.plot(thresholds_e5, accs_e5, marker="o", color="purple") # draw validation accuracy by threshold.
plt.title("Easy 5: validation chooses threshold") # title the plot.
plt.xlabel("threshold"); plt.ylabel("accuracy"); plt.ylim(0, 1.05); plt.show() # label and display.

▶ What you'll see: validation performance changes with threshold even though probabilities are fixed.

👀 Takeaway: threshold selection belongs to evaluation and deployment, not to the sigmoid formula alone.

## 🔴 Advanced

### Advanced 1 — Compare regularization strengths on train and validation loss

**Goal.** Sweep λ values and compare train versus validation log loss, because regularization is a stability knob that should be chosen with future-facing data. We build it in 5 steps.

In [ ]:
X_a1 = np.array([[0.0, 1.0], [0.4, 1.1], [0.8, 0.9], [1.6, 0.5], [2.1, 0.3], [2.8, 0.1], [3.2, 0.2], [3.5, 0.0]]) # create a small ordered dataset.
y_a1 = np.array([0., 0., 0., 1., 1., 1., 1., 1.]) # define labels.
train_idx_a1 = np.array([0, 1, 2, 3, 5, 6]) # choose training indices.
val_idx_a1 = np.array([4, 7]) # choose held-out validation indices.
print("train size:", len(train_idx_a1), "validation size:", len(val_idx_a1)) # inspect split sizes.

▶ What you'll see: six training examples and two validation examples.

In [ ]:
lams_a1 = np.array([0.0, 0.02, 0.1, 0.5, 1.0]) # define candidate regularization strengths.
train_losses_a1 = [] # store train losses.
val_losses_a1 = [] # store validation losses.
print("lambda grid:", lams_a1) # inspect the sweep values.

▶ What you'll see: the sweep ranges from no regularization to strong shrinkage.

In [ ]:
for lam_a1 in lams_a1: # train one model for each lambda.
    w_a1 = np.zeros(2) # reset weights for fair comparison.
    b_a1 = 0.0 # reset bias.
    Xtr_a1 = X_a1[train_idx_a1] # gather training features.
    ytr_a1 = y_a1[train_idx_a1] # gather training labels.
    for step_a1 in range(500): # run gradient descent.
        p_a1 = 1 / (1 + np.exp(-(Xtr_a1 @ w_a1 + b_a1))) # compute training probabilities.
        grad_w_a1 = Xtr_a1.T @ (p_a1 - ytr_a1) / len(ytr_a1) + lam_a1 * w_a1 # regularized gradient.
        grad_b_a1 = float(np.mean(p_a1 - ytr_a1)) # bias gradient.
        w_a1 -= 0.15 * grad_w_a1 # update weights.
        b_a1 -= 0.15 * grad_b_a1 # update bias.
    ptr_a1 = np.clip(1 / (1 + np.exp(-(Xtr_a1 @ w_a1 + b_a1))), 1e-12, 1 - 1e-12) # final train probabilities.
    pval_a1 = np.clip(1 / (1 + np.exp(-(X_a1[val_idx_a1] @ w_a1 + b_a1))), 1e-12, 1 - 1e-12) # final validation probabilities.
    train_losses_a1.append(float(np.mean(-(ytr_a1*np.log(ptr_a1) + (1-ytr_a1)*np.log(1-ptr_a1))))) # store train log loss.
    val_losses_a1.append(float(np.mean(-(y_a1[val_idx_a1]*np.log(pval_a1) + (1-y_a1[val_idx_a1])*np.log(1-pval_a1))))) # store validation log loss.
print("train losses:", np.round(train_losses_a1, 3)) # inspect train losses.
print("validation losses:", np.round(val_losses_a1, 3)) # inspect validation losses.

▶ What you'll see: train loss and validation loss do not necessarily prefer the same λ.

In [ ]:
best_lam_a1 = float(lams_a1[int(np.argmin(val_losses_a1))]) # select lambda with lowest validation loss.
print("best validation lambda:", best_lam_a1) # inspect selected regularization.
assert best_lam_a1 in lams_a1 # verify the selected value came from the grid.

▶ What you'll see: validation, not training loss alone, chooses the regularization setting.

In [ ]:
plt.figure(figsize=(5, 3)) # create the regularization sweep figure.
plt.plot(lams_a1, train_losses_a1, marker="o", label="train") # plot train loss.
plt.plot(lams_a1, val_losses_a1, marker="o", label="validation") # plot validation loss.
plt.axvline(best_lam_a1, color="red", linestyle="--", label="best λ") # mark selected lambda.
plt.title("Advanced 1: regularization sweep") # title the plot.
plt.xlabel("λ"); plt.ylabel("log loss"); plt.legend(); plt.show() # label and display.

▶ What you'll see: regularization trades off fitting the sample and stabilizing validation behavior.

👀 Takeaway: λ should be tuned on validation data because the lowest training loss can be too flexible.

### Advanced 2 — Visualize the learned probability surface

**Goal.** Plot probabilities across a two-feature plane, because logistic regression makes a smooth probability field around a linear boundary. We build it in 4 steps.

In [ ]:
X_a2 = np.array([[0.2, 1.2], [0.7, 1.1], [1.1, 0.9], [1.8, 0.6], [2.4, 0.4], [2.9, 0.2]]) # define two-dimensional examples.
y_a2 = np.array([0., 0., 0., 1., 1., 1.]) # define labels.
w_a2 = np.zeros(2) # initialize weights.
b_a2 = 0.0 # initialize bias.
print("dataset shape:", X_a2.shape) # inspect the tiny 2-D dataset.

▶ What you'll see: six examples in a two-dimensional feature space.

In [ ]:
for step_a2 in range(600): # fit logistic regression with mild regularization.
    p_a2 = 1 / (1 + np.exp(-(X_a2 @ w_a2 + b_a2))) # compute probabilities.
    grad_w_a2 = X_a2.T @ (p_a2 - y_a2) / len(y_a2) + 0.03 * w_a2 # compute regularized gradient.
    grad_b_a2 = float(np.mean(p_a2 - y_a2)) # compute bias gradient.
    w_a2 -= 0.2 * grad_w_a2 # update weights.
    b_a2 -= 0.2 * grad_b_a2 # update bias.
print("learned w:", np.round(w_a2, 3), "b:", round(b_a2, 3)) # inspect learned parameters.

▶ What you'll see: weights create a sloped linear boundary in the feature plane.

In [ ]:
x1_a2 = np.linspace(0, 3.2, 80) # grid x-axis.
x2_a2 = np.linspace(0, 1.5, 70) # grid y-axis.
xx_a2, yy_a2 = np.meshgrid(x1_a2, x2_a2) # create a 2-D grid.
grid_a2 = np.c_[xx_a2.ravel(), yy_a2.ravel()] # flatten grid points into rows.
pp_a2 = 1 / (1 + np.exp(-(grid_a2 @ w_a2 + b_a2))) # compute probabilities over the grid.
print("grid probability range:", round(float(pp_a2.min()), 3), "to", round(float(pp_a2.max()), 3)) # inspect range.

▶ What you'll see: probabilities vary smoothly from near 0 to near 1 across the plane.

In [ ]:
plt.figure(figsize=(5, 3.8)) # create a probability surface plot.
plt.contourf(xx_a2, yy_a2, pp_a2.reshape(xx_a2.shape), levels=20, cmap="RdYlBu_r", alpha=0.8) # draw probability contours.
plt.colorbar(label="p(y=1)") # add probability scale.
plt.contour(xx_a2, yy_a2, pp_a2.reshape(xx_a2.shape), levels=[0.5], colors="black") # draw decision boundary.
plt.scatter(X_a2[:, 0], X_a2[:, 1], c=y_a2, cmap="coolwarm", edgecolor="black", s=80) # overlay training points.
plt.title("Advanced 2: logistic probability surface") # title the figure.
plt.xlabel("x1"); plt.ylabel("x2"); plt.show() # label and display.

▶ What you'll see: curved probability colors surround a straight 0.5 decision boundary.

👀 Takeaway: sigmoid probabilities are nonlinear in color, but the default decision boundary remains a line.

### Advanced 3 — Compare calibrated log loss with accuracy

**Goal.** Show that two models can have the same hard-label accuracy but different log loss, because probability quality matters beyond thresholded decisions. We build it in 3 steps.

In [ ]:
y_a3 = np.array([0, 0, 1, 1]) # define four true labels.
p_confident_a3 = np.array([0.05, 0.10, 0.90, 0.95]) # define confident correct probabilities.
p_timids_a3 = np.array([0.45, 0.40, 0.60, 0.55]) # define timid but still correct probabilities.
print("hard predictions equal?", np.array_equal((p_confident_a3 >= 0.5).astype(int), (p_timids_a3 >= 0.5).astype(int))) # compare thresholded outputs.

▶ What you'll see: both probability sets make the same hard predictions at threshold 0.5.

In [ ]:
def_loss_conf_a3 = -(y_a3*np.log(p_confident_a3) + (1-y_a3)*np.log(1-p_confident_a3)) # compute confident losses.
def_loss_timid_a3 = -(y_a3*np.log(p_timids_a3) + (1-y_a3)*np.log(1-p_timids_a3)) # compute timid losses.
acc_conf_a3 = np.mean((p_confident_a3 >= 0.5).astype(int) == y_a3) # compute confident accuracy.
acc_timid_a3 = np.mean((p_timids_a3 >= 0.5).astype(int) == y_a3) # compute timid accuracy.
ll_conf_a3 = float(np.mean(def_loss_conf_a3)) # compute confident log loss.
ll_timid_a3 = float(np.mean(def_loss_timid_a3)) # compute timid log loss.
print("accuracies:", acc_conf_a3, acc_timid_a3) # inspect equal accuracies.
print("log losses:", round(ll_conf_a3, 3), round(ll_timid_a3, 3)) # inspect different probability quality.
assert acc_conf_a3 == acc_timid_a3 == 1.0 # verify same hard-label accuracy.
assert ll_conf_a3 < ll_timid_a3 # verify confident correct probabilities earn lower log loss.

▶ What you'll see: accuracy is identical, but log loss favors the better-calibrated confident model.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a metric comparison plot.
plt.bar(["confident log loss", "timid log loss"], [ll_conf_a3, ll_timid_a3], color=["seagreen", "orange"]) # compare losses.
plt.title("Advanced 3: same accuracy, different log loss") # title the plot.
plt.ylabel("mean log loss") # label metric scale.
plt.xticks(rotation=10); plt.show() # display with readable labels.

▶ What you'll see: the confident-correct model has much lower log loss despite equal accuracy.

👀 Takeaway: logistic regression's probability output should be evaluated with probability-aware metrics, not only hard-label accuracy.

### Advanced 4 — Show how feature scaling changes regularization

**Goal.** Compare the same information on different numeric scales, because L2 regularization penalizes coefficient size and therefore depends on feature scaling. We build it in 4 steps.

In [ ]:
x_raw_a4 = np.array([1., 2., 3., 4., 5., 6.]) # define one informative feature.
y_a4 = np.array([0., 0., 0., 1., 1., 1.]) # define labels.
X_big_a4 = x_raw_a4[:, None] * 100 # represent the feature on a large scale.
X_scaled_a4 = ((x_raw_a4 - x_raw_a4.mean()) / x_raw_a4.std())[:, None] # standardize the same feature.
print("big scale range:", X_big_a4.min(), X_big_a4.max()) # inspect unscaled range.
print("scaled mean/std:", round(float(X_scaled_a4.mean()), 3), round(float(X_scaled_a4.std()), 3)) # inspect standardized feature.

▶ What you'll see: the same ordering can be represented with very different numeric magnitudes.

In [ ]:
results_a4 = [] # store learned weights and losses for the two scales.
for X_cur_a4 in [X_big_a4, X_scaled_a4]: # train once on each feature scale.
    w_cur_a4 = np.zeros(1) # initialize one weight.
    b_cur_a4 = 0.0 # initialize bias.
    for step_a4 in range(700): # run gradient descent.
        p_cur_a4 = 1 / (1 + np.exp(-(X_cur_a4 @ w_cur_a4 + b_cur_a4))) # compute probabilities.
        grad_w_cur_a4 = X_cur_a4.T @ (p_cur_a4 - y_a4) / len(y_a4) + 0.2 * w_cur_a4 # regularized gradient.
        grad_b_cur_a4 = float(np.mean(p_cur_a4 - y_a4)) # bias gradient.
        w_cur_a4 -= 0.001 * grad_w_cur_a4 if X_cur_a4.max() > 10 else 0.2 * grad_w_cur_a4 # use a smaller safe step for huge scale.
        b_cur_a4 -= 0.001 * grad_b_cur_a4 if X_cur_a4.max() > 10 else 0.2 * grad_b_cur_a4 # update bias with matching scale.
    p_final_cur_a4 = np.clip(1 / (1 + np.exp(-(X_cur_a4 @ w_cur_a4 + b_cur_a4))), 1e-12, 1 - 1e-12) # final probabilities.
    loss_cur_a4 = float(np.mean(-(y_a4*np.log(p_final_cur_a4) + (1-y_a4)*np.log(1-p_final_cur_a4)))) # final log loss.
    results_a4.append((float(w_cur_a4[0]), b_cur_a4, loss_cur_a4)) # store summary.
print("weight/loss summaries:", [(round(r[0], 4), round(r[2], 3)) for r in results_a4]) # inspect learned weight scales.

▶ What you'll see: coefficients are not directly comparable when feature units differ.

In [ ]:
loss_big_a4 = results_a4[0][2] # read unscaled-feature loss.
loss_scaled_a4 = results_a4[1][2] # read standardized-feature loss.
print("big-scale loss:", round(loss_big_a4, 3), "scaled loss:", round(loss_scaled_a4, 3)) # compare fit quality.
assert np.isfinite(loss_big_a4) and np.isfinite(loss_scaled_a4) # verify both training runs stayed finite.

▶ What you'll see: both models train, but the regularized coefficient sizes live on different scales.

In [ ]:
plt.figure(figsize=(4.6, 3)) # create a scale comparison plot.
plt.bar(["big-scale |w|", "standardized |w|"], [abs(results_a4[0][0]), abs(results_a4[1][0])], color=["crimson", "teal"]) # compare absolute coefficient magnitudes.
plt.title("Advanced 4: scaling changes coefficient size") # title the plot.
plt.ylabel("absolute learned coefficient") # label coefficient scale.
plt.xticks(rotation=10); plt.show() # display the chart.

▶ What you'll see: the raw coefficient magnitude depends strongly on the units used for the feature.

👀 Takeaway: regularized logistic regression should use sensible feature scaling so penalties mean comparable things across coefficients.

### Advanced 5 — Diagnose the missing-cost pitfall in model selection

**Goal.** Reproduce the lesson's pitfall directly, because dropping the cost term changes which candidate looks best. We build it in 4 steps.

In [ ]:
raw_risk_a5 = np.array([0.272, 0.250, 0.300]) # define raw training risks for baseline, flexible, and stable candidates.
cost_a5 = np.array([0.070, 0.144, -0.026]) # define costs/adjustments that produce the lesson-style decision scores.
names_a5 = np.array(["baseline", "flexible", "stabilized"]) # label candidates.
print("raw risks:", dict(zip(names_a5, raw_risk_a5))) # inspect raw terms only.

▶ What you'll see: the flexible candidate has the prettiest raw training risk.

In [ ]:
full_scores_a5 = raw_risk_a5 + cost_a5 # compute the score that should drive selection.
raw_winner_a5 = names_a5[int(np.argmin(raw_risk_a5))] # choose incorrectly by raw fit alone.
full_winner_a5 = names_a5[int(np.argmin(full_scores_a5))] # choose correctly by full score.
print("full scores:", dict(zip(names_a5, np.round(full_scores_a5, 3)))) # inspect full decision scores.
print("raw winner:", raw_winner_a5, "full-score winner:", full_winner_a5) # compare decisions.
assert np.allclose(np.round(full_scores_a5, 3), [0.342, 0.394, 0.274]) # verify lesson decision scores.
assert raw_winner_a5 != full_winner_a5 # verify the pitfall changes the decision.

▶ What you'll see: optimizing raw risk alone picks flexible, while the full score picks stabilized.

In [ ]:
gap_a5 = full_scores_a5[1] - full_scores_a5[0] # compute flexible minus baseline gap.
rel_gap_a5 = gap_a5 / full_scores_a5[1] # compute relative gap.
print("baseline-vs-flexible gap:", round(gap_a5, 3), "relative:", round(rel_gap_a5, 3)) # inspect evidence gap.
assert round(gap_a5, 3) == 0.052 # verify lesson gap.
assert round(rel_gap_a5, 3) == 0.132 # verify relative gap.

▶ What you'll see: the same 0.052 gap appears once all candidates are on the proper scale.

In [ ]:
x_a5 = np.arange(len(names_a5)) # create grouped-bar positions.
plt.figure(figsize=(5, 3)) # create comparison plot.
plt.bar(x_a5 - 0.18, raw_risk_a5, width=0.36, label="raw risk", color="orange") # show raw-only scores.
plt.bar(x_a5 + 0.18, full_scores_a5, width=0.36, label="full score", color="teal") # show correct scores.
plt.xticks(x_a5, names_a5) # label candidates.
plt.title("Advanced 5: cost term changes selection") # title the plot.
plt.ylabel("score"); plt.legend(); plt.show() # label and display.

▶ What you'll see: the raw-risk ranking and full-score ranking disagree.

👀 Takeaway: logistic regression model selection must use the full score implied by the method, not an isolated training fragment.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Logistic regression turns a linear score into a calibrated probability through the sigmoid link.

Logistic regression keeps ERM concrete by optimizing class probabilities rather than raw labels. Its link function makes the output interpretable, but model selection still uses the lesson's raw term, cost, and validation gap.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def lesson_score(losses, cost, alternative):
    losses = np.asarray(losses, dtype=float)
    raw = round(float(losses.mean()), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    return {
        "losses": losses,
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
    }


def binary_logistic_train(x_tr, y_tr, lr=0.2, steps=900, l2=0.02):
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    weights = np.zeros(X.shape[1])
    y = y_tr.astype(float)
    for step in range(steps):
        logits = X @ weights
        probs = 1.0 / (1.0 + np.exp(-logits))
        grad = X.T @ (probs - y) / y.size
        grad[1:] = grad[1:] + l2 * weights[1:]
        weights = weights - lr * grad
    return weights


def binary_logistic_predict(weights, x_te):
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    probs = 1.0 / (1.0 + np.exp(-(X @ weights)))
    return (probs >= 0.5).astype(int)


def softmax_train(x_tr, y_tr, lr=0.15, steps=1100, l2=0.01):
    classes = np.unique(y_tr)
    mapping = {label: idx for idx, label in enumerate(classes)}
    y_idx = np.array([mapping[label] for label in y_tr])
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    W = np.zeros((X.shape[1], classes.size))
    Y = np.eye(classes.size)[y_idx]
    for step in range(steps):
        logits = X @ W
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(logits)
        probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        grad = X.T @ (probs - Y) / X.shape[0]
        grad[1:, :] = grad[1:, :] + l2 * W[1:, :]
        W = W - lr * grad
    return W, classes


def softmax_predict(model, x_te):
    W, classes = model
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    logits = X @ W
    return classes[np.argmax(logits, axis=1)]


def glm_predict(x_tr, y_tr, x_te):
    classes = np.unique(y_tr)
    if classes.size == 2:
        weights = binary_logistic_train(x_tr, y_tr)
        return binary_logistic_predict(weights, x_te)
    model = softmax_train(x_tr, y_tr)
    return softmax_predict(model, x_te)


def lda_qda_predict(x_tr, y_tr, x_te, mode="lda"):
    classes, counts = np.unique(y_tr, return_counts=True)
    if x_tr.shape[0] <= classes.size or counts.min() < 2:
        model = GaussianNB(var_smoothing=1e-8)
    elif mode == "qda":
        model = QuadraticDiscriminantAnalysis(reg_param=0.08)
    else:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def gda_predict(x_tr, y_tr, x_te):
    model = GaussianNB(var_smoothing=1e-8)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def sklearn_logistic_predict(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2500)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def classifier_metrics(rungs, predictor):
    rows = []
    for level, item in enumerate(rungs, start=1):
        name, X, y = item
        accuracy = clf_accuracy(predictor, X, y)
        rows.append({"level": level, "name": name, "accuracy": float(accuracy)})
    return rows


def plot_classifier_summary(rungs, rows, predictor):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, item, row in zip(axes[:5], rungs, rows):
        name, X, y = item
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        preds = predictor(x_tr_s, y_tr, x_te_s)
        ax.scatter(x_te_s[:, 0], x_te_s[:, 1], c=preds, s=14, cmap="viridis", alpha=0.8)
        ax.set_title(f"D{row['level']} acc={row['accuracy']:.2f}")
        ax.set_xlabel("feature 0")
        ax.set_ylabel("feature 1")
    axes[5].plot([row["level"] for row in rows], [row["accuracy"] for row in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_title("Accuracy vs ladder rung")
    axes[5].set_xlabel("D1 to D5")
    axes[5].set_ylabel("held-out accuracy")
    plt.tight_layout()
    plt.show()

def logistic_regression_method(losses=None, cost=0.070, alternative=0.394):
    if losses is None:
        losses = np.array([0.235, 0.109, 0.471])
    return lesson_score(losses, cost, alternative)


def softmax_multinomial_regression_method(losses=None, cost=0.080, alternative=0.401):
    if losses is None:
        losses = np.array([0.246, 0.122, 0.488])
    return lesson_score(losses, cost, alternative)


def generalized_linear_models_method(losses=None, cost=0.090, alternative=0.429):
    if losses is None:
        losses = np.array([0.257, 0.135, 0.505])
    return lesson_score(losses, cost, alternative)


def linear_quadratic_discriminant_analysis_method(losses=None, cost=0.100, alternative=0.457):
    if losses is None:
        losses = np.array([0.268, 0.148, 0.522])
    return lesson_score(losses, cost, alternative)


def gaussian_discriminant_analysis_method(losses=None, cost=0.050, alternative=0.361):
    if losses is None:
        losses = np.array([0.180, 0.070, 0.539])
    return lesson_score(losses, cost, alternative)

## The concept, built once (D1)

The lesson formula is

$$p(y=1\mid x)=\sigma(w^\top x)=\frac{1}{1+e^{-w^\top x}}$$

For D1, the verified per-example losses are 0.235, 0.109, 0.471. The empirical risk is the average, and the model-selection score is that raw term plus the lesson cost.

In [ ]:
result = logistic_regression_method()
print(result)
assert np.isclose(result["raw"], 0.272)
assert np.isclose(result["score"], 0.342)
assert np.isclose(result["gap"], 0.052)

The exact arithmetic is $R_S=(0.235, 0.109, 0.471)/3=0.272$, then $score=R_S+0.070=0.342$. The tempting alternative is 0.394, so the validation gap is $0.394-0.342=0.052$.

In [ ]:
stable_score = 0.80 * result["score"]
relative_gap = result["gap"] / result["alternative"]
print(f"stable={stable_score:.3f} relative_gap={relative_gap:.3f}")
assert stable_score < result["score"]
assert relative_gap > 0.0

## The dataset ladder

The same method now runs on D1 through D5. The printed preview shows shape, class balance or target scale, and a small sample before any fitting.

In [ ]:
rungs = clf_ladder()
for level, item in enumerate(rungs, start=1):
    name, X, y = item
    labels, counts = np.unique(y, return_counts=True)
    class_info = dict(zip(labels.tolist(), counts.tolist()))
    print(f"D{level}: {name} X={X.shape} classes={class_info}")
    print("sample X", np.round(X[:3, : min(3, X.shape[1])], 3))
    print("sample y", y[:3])

## Run the same method across D1-D5

The single headline metric for this lesson is accuracy. Classification topics also print the shared logistic baseline for a no-special-skill comparison.

In [ ]:
predictor = sklearn_logistic_predict
rows = classifier_metrics(rungs, predictor)
print("rung | accuracy | logistic baseline | dataset")
for row, item in zip(rows, rungs):
    name, X, y = item
    baseline = clf_accuracy(logistic_baseline, X, y)
    print(f"D{row['level']} | {row['accuracy']:.3f} | {baseline:.3f} | {row['name']}")
assert len(rows) == 5
assert all(0.0 <= row["accuracy"] <= 1.0 for row in rows)

## Results visualization

The closing figure has two parts: small multiples for each rung and one summary curve from D1 to D5.

In [ ]:
plot_classifier_summary(rungs, rows, predictor)

## Pitfall on the hardest rung

Pitfall: optimizing the raw term and forgetting the cost. The wrong check looks only at the raw D5 metric; the fix restores the lesson's cost and gap before selecting a winner.

In [ ]:
name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
main_preds = predictor(x_tr, y_tr, x_te)
base_preds = logistic_baseline(x_tr, y_tr, x_te)
main_acc = accuracy_score(y_te, main_preds)
base_acc = accuracy_score(y_te, base_preds)
lesson = logistic_regression_method()
raw_only = 1.0 - main_acc
full_score = raw_only + lesson["cost"]
alt_score = (1.0 - base_acc) + lesson["alternative"]
print(f"Raw-only D5 error={raw_only:.3f} baseline_error={1.0 - base_acc:.3f}")
print(f"Full score with lesson cost={full_score:.3f} alternative score={alt_score:.3f}")
print(f"Lesson cost={lesson['cost']:.3f} gap={lesson['gap']:.3f}")
print(f"Macro-F1 sanity={f1_score(y_te, main_preds, average='macro'):.3f}")
assert lesson["gap"] > 0.0
assert 0.0 <= full_score

## Evaluate it + Practice

- Compare the headline metric with the no-skill or logistic baseline before claiming improvement.
- Run a cheap sanity check: shuffled labels should damage accuracy, and injected outliers should make robust regression matter.
- Ablation: turn off the key idea, such as Huber clipping, softmax normalization, the GLM link, covariance modeling, or Bayes priors; the D5 metric should usually drop.
- Failure signals include unstable validation gaps, wildly different scales, singular covariance warnings, or a D5 score that only wins before cost is included.

Practice 1: change the cost term and recompute the decision score.

Practice 2: rerun D5 after removing one informative feature group and compare the metric.

Practice 3: create a shuffled-label baseline and explain why it should fail.